# Bayesian Hierarchical Analysis of CSD Waveform Features Before vs 48h After TBI

This notebook analyzes two CSV datasets:

1. **Pre-TBI CSD waveform features**
2. **48h post-TBI CSD waveform features**

Each row is assumed to represent one cell recorded during one CSD event. Each animal contributes approximately 20-100 cells, with one CSD event per animal.

Primary questions:

1. Does **AUC increase** 48 hours after TBI?
2. Does **total duration increase** 48 hours after TBI?
3. Does **amplitude decrease** 48 hours after TBI?

The model is hierarchical because cells are nested within animals:

```text
Condition
  └── Mouse
        └── Cell
```

## Libraries used

| Library | Purpose |
|---|---|
| `pathlib` | File and folder paths |
| `numpy` | Numerical arrays and posterior calculations |
| `pandas` | Import, clean, combine, and summarize CSVs |
| `matplotlib` | Posterior plots and summary figures |
| `seaborn` | Exploratory box/strip plots |
| `pymc` | Bayesian hierarchical modeling |
| `arviz` | Bayesian diagnostics, summaries, credible intervals |
| `warnings` | Suppress non-critical warnings |

Install if needed:

```bash
pip install numpy pandas matplotlib seaborn pymc arviz
```

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import pymc as pm
import arviz as az

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 30)

sns.set_context("notebook")
sns.set_style("whitegrid")

## Step 1 — Set paths

Expected CSV columns:

```text
animal_id, amplitude, total_duration, AUC, rise_time, decay_time
```

The notebook also accepts common alternate names such as `mouse_id`, `duration`, `auc`, `rise`, and `decay`.

In [ ]:
data_dir = Path("../data")

pre_csv = data_dir / "csd_waveforms_pre_tbi.csv"
post_csv = data_dir / "csd_waveforms_48h_post_tbi.csv"

output_dir = Path("../outputs/bayesian_tbi_csd")
fig_dir = output_dir / "figures"
model_dir = output_dir / "models"

output_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)

print("Pre-TBI CSV:", pre_csv)
print("Post-TBI CSV:", post_csv)

## Step 2 — Import and clean data

This step loads pre/post CSVs, adds condition labels, standardizes column names, and combines both datasets.

In [ ]:
def standardize_columns(df):
    rename_map = {}

    for col in df.columns:
        clean = col.strip().lower().replace(" ", "_").replace("-", "_")

        if clean in ["mouse", "mouse_id", "animal", "animal_id", "animalid"]:
            rename_map[col] = "animal_id"
        elif clean in ["amp", "amplitude", "peak_amplitude", "peak_dff"]:
            rename_map[col] = "amplitude"
        elif clean in ["duration", "total_duration", "total_duration_s", "total_duration_sec"]:
            rename_map[col] = "total_duration"
        elif clean in ["auc", "area_under_curve", "area"]:
            rename_map[col] = "AUC"
        elif clean in ["rise", "rise_time", "rise_time_s", "rise_sec"]:
            rename_map[col] = "rise_time"
        elif clean in ["decay", "decay_time", "decay_time_s", "decay_sec"]:
            rename_map[col] = "decay_time"
        else:
            rename_map[col] = clean

    return df.rename(columns=rename_map)


pre_df = standardize_columns(pd.read_csv(pre_csv))
post_df = standardize_columns(pd.read_csv(post_csv))

pre_df["condition"] = "pre_TBI"
post_df["condition"] = "post_TBI_48h"

df = pd.concat([pre_df, post_df], ignore_index=True)

required_cols = [
    "animal_id",
    "condition",
    "amplitude",
    "total_duration",
    "AUC",
    "rise_time",
    "decay_time"
]

missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

for col in ["amplitude", "total_duration", "AUC", "rise_time", "decay_time"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["animal_id"] = df["animal_id"].astype(str)
df = df.dropna(subset=required_cols).copy()
df["condition_code"] = (df["condition"] == "post_TBI_48h").astype(int)

print("Combined shape:", df.shape)
display(df.head())

## Step 3 — QC and descriptive summaries

This confirms sample sizes, animal counts, and cell counts per animal.

In [ ]:
feature_cols = ["amplitude", "total_duration", "AUC", "rise_time", "decay_time"]

print("Rows per condition:")
display(df["condition"].value_counts())

print("Animals per condition:")
display(df.groupby("condition")["animal_id"].nunique().rename("n_animals"))

print("Cells per animal:")
display(df.groupby(["condition", "animal_id"]).size().reset_index(name="n_cells"))

summary = df.groupby("condition")[feature_cols].agg(["count", "mean", "std", "median", "min", "max"]).round(4)
display(summary)
summary.to_csv(output_dir / "condition_feature_summary.csv")

In [ ]:
for feature in feature_cols:
    plt.figure(figsize=(7, 5))
    sns.boxplot(data=df, x="condition", y=feature)
    sns.stripplot(data=df, x="condition", y=feature, color="black", alpha=0.25, size=2)
    plt.title(f"{feature}: pre-TBI vs 48h post-TBI")
    plt.tight_layout()
    out_path = fig_dir / f"{feature}_condition_boxplot.tiff"
    plt.savefig(out_path, dpi=300, format="tiff", bbox_inches="tight")
    plt.show()

## Step 4 — Bayesian hierarchical model

For each feature, the model is:

```text
feature_cell = intercept + TBI_effect * post_TBI + animal_effect + noise
```

Where:

- `TBI_effect > 0` means the feature increased after TBI
- `TBI_effect < 0` means the feature decreased after TBI

The model estimates uncertainty over the TBI effect while accounting for animal-to-animal variability.

In [ ]:
def fit_hierarchical_feature_model(data, feature, draws=2000, tune=2000, target_accept=0.9, random_seed=42):
    model_df = data[["animal_id", "condition_code", feature]].dropna().copy()
    y_raw = model_df[feature].values.astype(float)

    y_mean = y_raw.mean()
    y_std = y_raw.std()
    if y_std == 0:
        raise ValueError(f"{feature} has zero variance.")

    y = (y_raw - y_mean) / y_std
    animal_codes, animal_index = pd.factorize(model_df["animal_id"])
    condition_code = model_df["condition_code"].values.astype(int)

    coords = {"animal": animal_index, "obs": np.arange(len(model_df))}

    with pm.Model(coords=coords) as model:
        animal_idx = pm.Data("animal_idx", animal_codes, dims="obs")
        post_tbi = pm.Data("post_tbi", condition_code, dims="obs")

        intercept = pm.Normal("intercept", mu=0, sigma=1)
        beta_tbi = pm.Normal("beta_tbi", mu=0, sigma=1)

        sigma_animal = pm.HalfNormal("sigma_animal", sigma=1)
        animal_offset = pm.Normal("animal_offset", mu=0, sigma=1, dims="animal")
        animal_effect = pm.Deterministic("animal_effect", animal_offset * sigma_animal, dims="animal")

        sigma = pm.HalfNormal("sigma", sigma=1)
        mu = intercept + beta_tbi * post_tbi + animal_effect[animal_idx]

        likelihood = pm.Normal("likelihood", mu=mu, sigma=sigma, observed=y, dims="obs")

        trace = pm.sample(
            draws=draws,
            tune=tune,
            target_accept=target_accept,
            random_seed=random_seed,
            return_inferencedata=True
        )

    metadata = {
        "feature": feature,
        "y_mean": y_mean,
        "y_std": y_std,
        "n_cells": len(model_df),
        "n_animals": len(animal_index)
    }
    return trace, metadata

## Step 5 — Fit one model per waveform feature

Primary hypotheses:

| Feature | Expected after 48h TBI |
|---|---|
| AUC | Increase |
| total_duration | Increase |
| amplitude | Decrease |
| rise_time | Exploratory |
| decay_time | Exploratory |

In [ ]:
features_to_model = ["amplitude", "total_duration", "AUC", "rise_time", "decay_time"]

traces = {}
model_metadata = {}

for feature in features_to_model:
    print("\nFitting model for:", feature)
    trace, metadata = fit_hierarchical_feature_model(
        df,
        feature=feature,
        draws=2000,
        tune=2000,
        target_accept=0.9,
        random_seed=42
    )
    traces[feature] = trace
    model_metadata[feature] = metadata
    az.to_netcdf(trace, model_dir / f"{feature}_hierarchical_model_trace.nc")

print("Finished all models.")

## Step 6 — Posterior summaries and directional probabilities

Key Bayesian outputs:

- `P(effect > 0)` = probability that the feature increased after TBI
- `P(effect < 0)` = probability that the feature decreased after TBI
- 95% credible interval = likely range of the effect given the data and model

Primary interpretation:

- For AUC and duration, look for high `P(effect > 0)`
- For amplitude, look for high `P(effect < 0)`

In [ ]:
summary_rows = []

for feature, trace in traces.items():
    beta = trace.posterior["beta_tbi"].values.flatten()
    y_std = model_metadata[feature]["y_std"]
    effect_original_units = beta * y_std

    prob_increase = np.mean(effect_original_units > 0)
    prob_decrease = np.mean(effect_original_units < 0)
    hdi = az.hdi(effect_original_units, hdi_prob=0.95)

    summary_rows.append({
        "feature": feature,
        "posterior_mean_effect_original_units": effect_original_units.mean(),
        "posterior_median_effect_original_units": np.median(effect_original_units),
        "hdi_2.5": hdi[0],
        "hdi_97.5": hdi[1],
        "probability_increase_after_TBI": prob_increase,
        "probability_decrease_after_TBI": prob_decrease,
        "n_cells": model_metadata[feature]["n_cells"],
        "n_animals": model_metadata[feature]["n_animals"]
    })

posterior_summary = pd.DataFrame(summary_rows)
posterior_summary_path = output_dir / "bayesian_tbi_effect_posterior_summary.csv"
posterior_summary.to_csv(posterior_summary_path, index=False)
display(posterior_summary)
print("Saved:", posterior_summary_path)

In [ ]:
hypothesis_rows = []

for _, row in posterior_summary.iterrows():
    feature = row["feature"]

    if feature in ["AUC", "total_duration"]:
        expected_direction = "increase"
        probability_supporting_hypothesis = row["probability_increase_after_TBI"]
    elif feature == "amplitude":
        expected_direction = "decrease"
        probability_supporting_hypothesis = row["probability_decrease_after_TBI"]
    else:
        expected_direction = "exploratory"
        probability_supporting_hypothesis = np.nan

    hypothesis_rows.append({
        "feature": feature,
        "expected_direction_after_TBI": expected_direction,
        "probability_supporting_hypothesis": probability_supporting_hypothesis,
        "posterior_mean_effect_original_units": row["posterior_mean_effect_original_units"],
        "hdi_2.5": row["hdi_2.5"],
        "hdi_97.5": row["hdi_97.5"]
    })

hypothesis_summary = pd.DataFrame(hypothesis_rows)
hypothesis_path = output_dir / "bayesian_directional_hypothesis_summary.csv"
hypothesis_summary.to_csv(hypothesis_path, index=False)
display(hypothesis_summary)
print("Saved:", hypothesis_path)

## Step 7 — Plot posterior TBI effects

A posterior distribution mostly above zero indicates evidence for increase. A posterior distribution mostly below zero indicates evidence for decrease.

In [ ]:
for feature, trace in traces.items():
    beta = trace.posterior["beta_tbi"].values.flatten()
    y_std = model_metadata[feature]["y_std"]
    effect_original_units = beta * y_std

    plt.figure(figsize=(8, 5))
    plt.hist(effect_original_units, bins=50, density=True, alpha=0.75)
    plt.axvline(0, color="black", linestyle="--", label="No effect")
    plt.axvline(effect_original_units.mean(), color="red", linestyle="-", label="Posterior mean")
    plt.xlabel(f"48h post-TBI effect on {feature}")
    plt.ylabel("Posterior density")
    plt.title(f"Posterior TBI Effect: {feature}")
    plt.legend()
    plt.tight_layout()
    out_path = fig_dir / f"{feature}_posterior_tbi_effect.tiff"
    plt.savefig(out_path, dpi=300, format="tiff", bbox_inches="tight")
    plt.show()
    print("Saved:", out_path)

## Step 8 — Forest plot of TBI effects

In [ ]:
plot_df = posterior_summary.sort_values("posterior_mean_effect_original_units")

plt.figure(figsize=(8, 5))
y_pos = np.arange(len(plot_df))
means = plot_df["posterior_mean_effect_original_units"].values
lower = plot_df["hdi_2.5"].values
upper = plot_df["hdi_97.5"].values

plt.errorbar(means, y_pos, xerr=[means - lower, upper - means], fmt="o", capsize=5)
plt.axvline(0, color="black", linestyle="--")
plt.yticks(y_pos, plot_df["feature"])
plt.xlabel("Estimated 48h post-TBI effect in original units")
plt.title("Bayesian Hierarchical TBI Effects")
plt.tight_layout()

forest_path = fig_dir / "bayesian_tbi_effect_forest_plot.tiff"
plt.savefig(forest_path, dpi=300, format="tiff", bbox_inches="tight")
plt.show()
print("Saved:", forest_path)

## Step 9 — MCMC diagnostics

Check:

- `r_hat` close to 1.00
- sufficient effective sample size
- trace plots without drifting trends

In [ ]:
for feature, trace in traces.items():
    print("\nDiagnostics for:", feature)
    display(az.summary(trace, var_names=["intercept", "beta_tbi", "sigma_animal", "sigma"], round_to=4))
    az.plot_trace(trace, var_names=["intercept", "beta_tbi", "sigma_animal", "sigma"])
    plt.tight_layout()
    trace_path = fig_dir / f"{feature}_trace_diagnostics.tiff"
    plt.savefig(trace_path, dpi=300, format="tiff", bbox_inches="tight")
    plt.show()
    print("Saved:", trace_path)

## Interpretation template

Example language:

```text
The Bayesian hierarchical model estimated a posterior probability of 0.96 that AUC is increased 48h after TBI. The posterior mean increase was X units with a 95% credible interval from lower to upper.
```

Suggested interpretation:

| Posterior probability | Interpretation |
|---|---|
| > 0.95 | Strong evidence |
| 0.90-0.95 | Moderate evidence |
| 0.75-0.90 | Weak/suggestive evidence |
| ~0.50 | Little directional evidence |

This model gives probability of the effect direction given the data and model assumptions.